![pageindex_banner](https://pageindex.ai/static/images/pageindex_banner.jpg)

<p align="center"><i>RAG на основе рассуждений&nbsp; ✧ &nbsp;без векторной БД&nbsp; ✧ &nbsp;без чанков&nbsp; ✧ &nbsp;извлечение как у человека</i></p>

<p align="center">
  <a href="https://vectify.ai">🏠 Домашняя страница</a>&nbsp; • &nbsp;
  <a href="https://dash.pageindex.ai">🖥️ Дашборд</a>&nbsp; • &nbsp;
  <a href="https://docs.pageindex.ai/quickstart">📚 Документация API</a>&nbsp; • &nbsp;
  <a href="https://github.com/VectifyAI/PageIndex">📦 GitHub</a>&nbsp; • &nbsp;
  <a href="https://discord.com/invite/VuXuf29EUj">💬 Discord</a>&nbsp; • &nbsp;
  <a href="https://ii2abc2jejf.typeform.com/to/tK3AXl8T">✉️ Контакты</a>&nbsp;
</p>

<div align="center">

[![Поставьте звезду на GitHub](https://img.shields.io/github/stars/VectifyAI/PageIndex?style=for-the-badge&logo=github&label=⭐️%20Star%20Us)](https://github.com/VectifyAI/PageIndex) &nbsp;&nbsp; [![Подписаться в X](https://img.shields.io/badge/Follow%20Us-000000?style=for-the-badge&logo=x&logoColor=white)](https://twitter.com/VectifyAI)

</div>

---


# Простой RAG без векторов с PageIndex


## Введение в PageIndex
PageIndex — новый фреймворк **RAG на основе рассуждений** и **без векторов**, который выполняет извлечение в два шага:  
1. Генерирует древовидную структуру (индекс) документа  
2. Выполняет извлечение на основе рассуждений через поиск по дереву  

<div align="center">
  <img src="https://docs.pageindex.ai/images/cookbook/vectorless-rag.png" width="70%">
</div>

По сравнению с традиционным векторным RAG, PageIndex предлагает:
- **Без векторной БД**: использует структуру документа и рассуждения LLM для извлечения.
- **Без чанков**: документы организованы в естественные разделы, а не искусственные фрагменты.
- **Извлечение как у человека**: имитирует навигацию экспертов по сложным документам.
- **Прозрачный процесс извлечения**: извлечение основано на рассуждениях — меньше приближенного семантического поиска («vibe retrieval»).

**Исследовательская заметка.** Дерево служит интерпретируемым промежуточным представлением: это позволяет отдельно оценивать качество индекса и качество поиска. В экспериментах фиксируйте параметры построения (макс. страниц на узел, лимиты токенов) и сравнивайте устойчивость структуры.


## 📝 Обзор ноутбука

Этот ноутбук показывает простой минимальный пример **RAG без векторов** с PageIndex. Вы узнаете, как:
- [x] Построить дерево PageIndex для документа
- [x] Выполнить извлечение на основе рассуждений с поиском по дереву
- [x] Сгенерировать ответы на основе извлеченного контекста

> ⚡ Примечание: это **минимальный пример**, который иллюстрирует базовую идею PageIndex, а не полный набор возможностей. Более продвинутые примеры будут опубликованы позже.

---

**Исследовательский фокус.** Минимальный сценарий удобен для абляций: можно сравнивать разные модели, отключать резюме узлов или менять глубину дерева, чтобы оценить вклад каждой части.


## Шаг 0: Подготовка


#### 0.1 Установка PageIndex


In [ ]:
%pip install -q --upgrade pageindex

#### 0.2 Настройка PageIndex


In [ ]:
from pageindex import PageIndexClient
import pageindex.utils as utils

# Get your PageIndex API key from https://dash.pageindex.ai/api-keys
PAGEINDEX_API_KEY = "YOUR_PAGEINDEX_API_KEY"
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

#### 0.3 Настройка LLM
Выберите предпочитаемую LLM для извлечения на основе рассуждений. В этом примере мы используем OpenAI GPT-4.1.


In [ ]:
import openai
OPENAI_API_KEY = "YOUR_OPENAI_API_KEY"

async def call_llm(prompt, model="gpt-4.1", temperature=0):
    client = openai.AsyncOpenAI(api_key=OPENAI_API_KEY)
    response = await client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature
    )
    return response.choices[0].message.content.strip()

## Шаг 1: Генерация дерева PageIndex


#### 1.1 Отправить документ для генерации дерева PageIndex


In [ ]:
import os, requests

# You can also use our GitHub repo to generate PageIndex tree
# https://github.com/VectifyAI/PageIndex

pdf_url = "https://arxiv.org/pdf/2501.12948.pdf"
pdf_path = os.path.join("../data", pdf_url.split('/')[-1])
os.makedirs(os.path.dirname(pdf_path), exist_ok=True)

response = requests.get(pdf_url)
with open(pdf_path, "wb") as f:
    f.write(response.content)
print(f"Downloaded {pdf_url}")

doc_id = pi_client.submit_document(pdf_path)["doc_id"]
print('Document Submitted:', doc_id)

Downloaded https://arxiv.org/pdf/2501.12948.pdf
Document Submitted: pi-cmeseq08w00vt0bo3u6tr244g


#### 1.2 Получить сгенерированную структуру дерева PageIndex


In [ ]:
if pi_client.is_retrieval_ready(doc_id):
    tree = pi_client.get_tree(doc_id, node_summary=True)['result']
    print('Simplified Tree Structure of the Document:')
    utils.print_tree(tree)
else:
    print("Processing document, please try again later...")

Simplified Tree Structure of the Document:
[{'title': 'DeepSeek-R1: Incentivizing Reasoning Cap...',
  'node_id': '0000',
  'prefix_summary': '# DeepSeek-R1: Incentivizing Reasoning C...',
  'nodes': [{'title': 'Abstract',
             'node_id': '0001',
             'summary': 'The partial document introduces two reas...'},
            {'title': 'Contents',
             'node_id': '0002',
             'summary': 'This partial document provides a detaile...'},
            {'title': '1. Introduction',
             'node_id': '0003',
             'prefix_summary': 'The partial document introduces recent a...',
             'nodes': [{'title': '1.1. Contributions',
                        'node_id': '0004',
                        'summary': 'This partial document outlines the main ...'},
                       {'title': '1.2. Summary of Evaluation Results',
                        'node_id': '0005',
                        'summary': 'The partial document provides a summary ...'}]},
    

## Шаг 2: Извлечение на основе рассуждений с поиском по дереву


#### 2.1 Использовать LLM для поиска по дереву и определить узлы, которые могут содержать релевантный контекст


In [21]:
import json

query = "What are the conclusions in this document?"

tree_without_text = utils.remove_fields(tree.copy(), fields=['text'])

search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

tree_search_result = await call_llm(search_prompt)

#### 2.2 Вывести найденные узлы и процесс рассуждения


In [57]:
node_map = utils.create_node_mapping(tree)
tree_search_result_json = json.loads(tree_search_result)

print('Reasoning Process:')
utils.print_wrapped(tree_search_result_json['thinking'])

print('\nRetrieved Nodes:')
for node_id in tree_search_result_json["node_list"]:
    node = node_map[node_id]
    print(f"Node ID: {node['node_id']}\t Page: {node['page_index']}\t Title: {node['title']}")

Reasoning Process:
The question asks for the conclusions in the document. Typically, conclusions are found in sections
explicitly titled 'Conclusion' or in sections summarizing the findings and implications of the work.
In this document tree, node 0019 ('5. Conclusion, Limitations, and Future Work') is the most
directly relevant, as it is dedicated to the conclusion and related topics. Additionally, the
'Abstract' (node 0001) may contain a high-level summary that sometimes includes concluding remarks,
but it is less likely to contain the full conclusions. Other sections like 'Discussion' (node 0018)
may discuss implications but are not explicitly conclusions. Therefore, the primary node is 0019.

Retrieved Nodes:
Node ID: 0019	 Page: 16	 Title: 5. Conclusion, Limitations, and Future Work


## Шаг 3: Генерация ответа


#### 3.1 Извлечь релевантный контекст из найденных узлов


In [58]:
node_list = json.loads(tree_search_result)["node_list"]
relevant_content = "\n\n".join(node_map[node_id]["text"] for node_id in node_list)

print('Retrieved Context:\n')
utils.print_wrapped(relevant_content[:1000] + '...')

Retrieved Context:

## 5. Conclusion, Limitations, and Future Work

In this work, we share our journey in enhancing model reasoning abilities through reinforcement
learning. DeepSeek-R1-Zero represents a pure RL approach without relying on cold-start data,
achieving strong performance across various tasks. DeepSeek-R1 is more powerful, leveraging cold-
start data alongside iterative RL fine-tuning. Ultimately, DeepSeek-R1 achieves performance
comparable to OpenAI-o1-1217 on a range of tasks.

We further explore distillation the reasoning capability to small dense models. We use DeepSeek-R1
as the teacher model to generate 800K training samples, and fine-tune several small dense models.
The results are promising: DeepSeek-R1-Distill-Qwen-1.5B outperforms GPT-4o and Claude-3.5-Sonnet on
math benchmarks with $28.9 \%$ on AIME and $83.9 \%$ on MATH. Other dense models also achieve
impressive results, significantly outperforming other instructiontuned models based on the same
underlying che

#### 3.2 Сгенерировать ответ на основе извлеченного контекста


In [59]:
answer_prompt = f"""
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a clear, concise answer based only on the context provided.
"""

print('Generated Answer:\n')
answer = await call_llm(answer_prompt)
utils.print_wrapped(answer)

Generated Answer:

The conclusions in this document are:

- DeepSeek-R1-Zero, a pure reinforcement learning (RL) approach without cold-start data, achieves
strong performance across various tasks.
- DeepSeek-R1, which combines cold-start data with iterative RL fine-tuning, is more powerful and
achieves performance comparable to OpenAI-o1-1217 on a range of tasks.
- Distilling DeepSeek-R1’s reasoning capabilities into smaller dense models is promising; for
example, DeepSeek-R1-Distill-Qwen-1.5B outperforms GPT-4o and Claude-3.5-Sonnet on math benchmarks,
and other dense models also show significant improvements over similar instruction-tuned models.

These results demonstrate the effectiveness of the RL-based approach and the potential for
distilling reasoning abilities into smaller models.


---

## 🎯 Что дальше

Этот ноутбук показал **базовый**, **минимальный** пример **RAG на основе рассуждений** и **без векторов** с PageIndex. Процесс иллюстрирует ключевую идею:
> *Построение иерархического дерева из документа, рассуждение по этому дереву и извлечение релевантного контекста без векторной БД или top-k поиска по сходству*.

Хотя здесь показан минимальный сценарий, фреймворк PageIndex рассчитан на **гораздо более продвинутые** кейсы. В следующих туториалах мы рассмотрим:
* **Многоузловое рассуждение с извлечением контента** — масштабирование поиска по дереву для извлечения и выбора релевантного контента из нескольких узлов.
* **Поиск по нескольким документам** — навигация на основе рассуждений по коллекциям документов, выходящая за рамки одного файла.
* **Эффективный поиск по дереву** — повышение эффективности поиска по дереву для длинных документов с большим числом узлов.
* **Интеграция экспертных знаний и выравнивание предпочтений** — добавление знаний прямо в LLM-поиск по дереву без дообучения.


## 🔎 Узнать больше о PageIndex
  <a href="https://vectify.ai">🏠 Домашняя страница</a>&nbsp; • &nbsp;
  <a href="https://dash.pageindex.ai">🖥️ Дашборд</a>&nbsp; • &nbsp;
  <a href="https://docs.pageindex.ai/quickstart">📚 Документация API</a>&nbsp; • &nbsp;
  <a href="https://github.com/VectifyAI/PageIndex">📦 GitHub</a>&nbsp; • &nbsp;
  <a href="https://discord.com/invite/VuXuf29EUj">💬 Discord</a>&nbsp; • &nbsp;
  <a href="https://ii2abc2jejf.typeform.com/to/tK3AXl8T">✉️ Контакты</a>

<br>

© 2025 [Vectify AI](https://vectify.ai)
